# Code: Evolutionary Distances

<!-- launch-badges -->
[![KTH JupyterHub](https://img.shields.io/badge/launch-KTH%20JupyterHub-F37626?logo=jupyter&logoColor=white)](https://193.10.159.40.nip.io/hub/user-redirect/git-pull?repo=https://github.com/statisticalbiotechnology/bibook&urlpath=lab/tree/bibook/bibook/phylogenetics/distance/distances_code.ipynb&branch=main)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/statisticalbiotechnology/bibook/blob/main/bibook/phylogenetics/distance/distances_code.ipynb)
[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/statisticalbiotechnology/bibook/main?labpath=bibook/phylogenetics/distance/distances_code.ipynb)

In this notebook we calculate distances from an existing multiple sequence alignment. We compare observed p-distances with Jukes–Cantor and Kimura two-parameter corrections, examine substitution saturation, and connect distance to time under a molecular clock. Tree construction is deliberately left for the computer lab.

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

alignment = {
    'Seq1': 'AAATCC-TATT',
    'Seq2': 'AAAACCGTAAT',
    'Seq3': 'TAGTCCGTAAT',
    'Seq4': 'TAGTCCGTCAT',
}

for name, sequence in alignment.items():
    print(f'{name}: {sequence}')

## Count comparable positions and substitutions

We exclude a column for a pair whenever either sequence contains a gap. A transition is $A \leftrightarrow G$ or $C \leftrightarrow T$; all other nucleotide mismatches are transversions.

In [ ]:
transition_pairs = {frozenset(('A', 'G')), frozenset(('C', 'T'))}

def substitution_counts(seq_a, seq_b):
    if len(seq_a) != len(seq_b):
        raise ValueError('Sequences must already be aligned')
    compared = transitions = transversions = 0
    for a, b in zip(seq_a.upper(), seq_b.upper()):
        if '-' in (a, b):
            continue
        compared += 1
        if a != b:
            if frozenset((a, b)) in transition_pairs:
                transitions += 1
            else:
                transversions += 1
    return compared, transitions, transversions

L, transitions, transversions = substitution_counts(alignment['Seq1'], alignment['Seq2'])
print(f'Compared sites: {L}')
print(f'Transitions: {transitions}; transversions: {transversions}')
print(f'p-distance: {(transitions + transversions) / L:.3f}')

## Three nucleotide distances

The corrected distances can be undefined for strongly diverged sequences. Returning `nan` makes this limitation visible instead of allowing an invalid logarithm.

In [ ]:
def nucleotide_distances(seq_a, seq_b):
    L, ts, tv = substitution_counts(seq_a, seq_b)
    if L == 0:
        return np.nan, np.nan, np.nan
    p = (ts + tv) / L
    P, Q = ts / L, tv / L
    jc_term = 1 - 4 * p / 3
    d_jc = -0.75 * math.log(jc_term) if jc_term > 0 else np.nan
    k2p_a, k2p_b = 1 - 2 * P - Q, 1 - 2 * Q
    d_k2p = (-0.5 * math.log(k2p_a) - 0.25 * math.log(k2p_b)
             if k2p_a > 0 and k2p_b > 0 else np.nan)
    return p, d_jc, d_k2p

p, jc, k2p = nucleotide_distances(alignment['Seq1'], alignment['Seq3'])
print(f'p = {p:.3f}, Jukes-Cantor = {jc:.3f}, Kimura 2P = {k2p:.3f}')

## Construct pairwise distance matrices

A distance matrix is symmetric and has zeroes on its diagonal. Notice that corrected distances are at least as large as the observed p-distances for these sequences.

In [ ]:
def distance_matrices(sequences):
    names = list(sequences)
    matrices = [np.zeros((len(names), len(names))) for _ in range(3)]
    for i in range(len(names)):
        for j in range(i + 1, len(names)):
            values = nucleotide_distances(sequences[names[i]], sequences[names[j]])
            for matrix, value in zip(matrices, values):
                matrix[i, j] = matrix[j, i] = value
    return names, matrices

names, (p_matrix, jc_matrix, k2p_matrix) = distance_matrices(alignment)
for label, matrix in [('p-distance', p_matrix), ('Jukes-Cantor', jc_matrix), ('Kimura 2P', k2p_matrix)]:
    print(f'\n{label}')
    print('       ' + ''.join(f'{name:>8}' for name in names))
    for name, row in zip(names, matrix):
        print(f'{name:<7}' + ''.join(f'{value:8.3f}' for value in row))

## Visualize substitution saturation

Under the Jukes–Cantor model, the expected observed difference approaches 0.75 even while the evolutionary distance continues to increase. Multiple substitutions have erased information about earlier changes.

In [ ]:
true_distance = np.linspace(0, 3, 300)
expected_p = 0.75 * (1 - np.exp(-4 * true_distance / 3))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(true_distance, expected_p, label='Expected p-distance')
ax.plot(true_distance, true_distance, '--', color='grey', label='No multiple hits')
ax.axhline(0.75, color='black', linestyle=':', label='Saturation limit')
ax.set(xlabel='Actual substitutions per site', ylabel='Observed fraction different',
       title='Substitution saturation under Jukes–Cantor', ylim=(0, 1))
ax.legend();

## Distance and time under a molecular clock

For two descendant lineages evolving at the same rate, $d=2rt$. The factor two appears because substitutions accumulate independently along both branches.

In [ ]:
def divergence_time(distance, rate):
    return distance / (2 * rate)

distance = 0.12       # substitutions per site between two sequences
rate = 1e-8           # substitutions per site per year along each lineage
time_years = divergence_time(distance, rate)
print(f'Estimated divergence time: {time_years / 1e6:.1f} million years')

rates = np.array([0.5e-8, 1e-8, 2e-8])
for r in rates:
    print(f'r = {r:.1e}: t = {divergence_time(distance, r) / 1e6:.1f} Myr')

## Exercises

1. Add a fifth aligned sequence and inspect how the three distance matrices change.
2. Construct a pair for which the p-distance is unchanged but the Kimura distance differs because transitions and transversions occur in different proportions.
3. Increase the observed distance in the molecular-clock example. Which assumptions must remain valid for the time estimate to be meaningful?
4. Why do the Jukes–Cantor and Kimura functions return `nan` for some highly divergent sequence pairs?